In [1]:
import pyarrow.dataset as ds
import os
ROOT = '/Volumes/Backup Plus/Zaman/graph/data'
edges = ['edge_rek_debit', 'edge_rek_credit', 'edge_nasabah_memiliki_simp', 'edge_nasabah_memiliki_pinj', 'edge_nasabah_is_pekerja']
for e in edges:
    path = os.path.join(ROOT, e)
    if os.path.exists(path):
        d = ds.dataset(path, format='parquet')
        print(f'{e}: {d.schema.names}')
    else:
        print(f'{e}: NOT FOUND')


edge_rek_debit: ['src', 'dst', 'relation']
edge_rek_credit: ['src', 'dst', 'relation']
edge_nasabah_memiliki_simp: ['src', 'dst', 'relation']
edge_nasabah_memiliki_pinj: ['src', 'dst', 'relation']
edge_nasabah_is_pekerja: ['src', 'dst', 'relation']


In [1]:
import lmdb

# Check transaksi LMDB
lmdb_path = "/Volumes/Backup Plus/Zaman/graph/lmdb_node_mapping/transaksi.lmdb"
env = lmdb.open(lmdb_path, readonly=True, lock=False)

with env.begin() as txn:
    stats = txn.stat()
    print(f"Transaksi LMDB stats: {stats}")
    
    # Get first 5 entries
    cursor = txn.cursor()
    count = 0
    for key, value in cursor:
        print(f"  {key.decode()} -> {value.decode()}")
        count += 1
        if count >= 5:
            break

env.close()

Transaksi LMDB stats: {'psize': 16384, 'depth': 0, 'branch_pages': 0, 'leaf_pages': 0, 'overflow_pages': 0, 'entries': 0}


In [2]:
# Debug: Compare ID formats between node maps and edge files
import pyarrow.parquet as pq
import os

DATA_DIR = "/Volumes/Backup Plus/Zaman/graph/data"

# Sample edge file IDs
edge_file = os.path.join(DATA_DIR, "edge_nasabah_is_pekerja")
pf = pq.ParquetFile(f"{edge_file}/ds=202308.parquet")  # or whatever file exists
batch = next(pf.iter_batches(batch_size=5, columns=['src', 'dst']))

print("Edge file sample (src=cif, dst=pn):")
print(f"  src: {batch['src'].to_pylist()}")
print(f"  dst: {batch['dst'].to_pylist()}")

# Sample from node maps
print("\nNode map sample:")
print(f"  nasabah (first 3): {list(NODE_MAPS['nasabah'].keys())[:3]}")
print(f"  pekerja (first 3): {list(NODE_MAPS['pekerja'].keys())[:3]}")

# Check types
src_sample = str(batch['src'].to_pylist()[0])
print(f"\nEdge src type: {type(batch['src'].to_pylist()[0])}, value: '{src_sample}'")
print(f"Node map key type: {type(list(NODE_MAPS['nasabah'].keys())[0])}")

# Check if src exists in node map
print(f"\nDoes '{src_sample}' exist in nasabah map? {src_sample in NODE_MAPS['nasabah']}")

Edge file sample (src=cif, dst=pn):
  src: ['SJ69172', 'I521632', 'CB81646', 'R523993', 'R329060']
  dst: [26154, 26811, 27812, 27981, 54881]

Node map sample:


NameError: name 'NODE_MAPS' is not defined

In [3]:
import pyarrow.parquet as pq
import os
import glob

DATA_DIR = "/Volumes/Backup Plus/Zaman/graph/data"

# Check credit edge file
credit_dir = os.path.join(DATA_DIR, "edge_rek_credit")
files = glob.glob(f"{credit_dir}/*.parquet")
print(f"Credit files: {len(files)}")

if files:
    pf = pq.ParquetFile(files[0])
    print(f"Schema: {pf.schema_arrow.names}")
    
    batch = next(pf.iter_batches(batch_size=5, columns=['src', 'dst']))
    print(f"\nSample src: {batch['src'].to_pylist()}")
    print(f"Sample dst: {batch['dst'].to_pylist()}")
    
    # Check if these exist in node maps
    src_sample = str(batch['src'].to_pylist()[0])
    dst_sample = str(batch['dst'].to_pylist()[0])
    
    print(f"\n'{src_sample}' in simpanan? {src_sample in NODE_MAPS['simpanan']}")
    print(f"'{dst_sample}' in transaksi? {dst_sample in NODE_MAPS['transaksi']}")

Credit files: 13
Schema: ['src', 'dst', 'relation']

Sample src: ['8888556300128506779101011984532150000000', '852216928181785067791010046445235650000039101001728995', '9901128710000185084777010085935351000000039101000638993', '8888239906251485084777010150035392000000039101000656991', '88883097471850635701181046508300000000']
Sample dst: [Decimal('779101011984532'), Decimal('779101004644523'), Decimal('477701008593535'), Decimal('477701015003539'), Decimal('35701181046508')]


NameError: name 'NODE_MAPS' is not defined